### ETL Pipeline

In [0]:
import os
import urllib.request

from pyspark.sql import functions as F
from pyspark.sql import types as T

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

SNAPSHOT = "2026-09-17"

CATALOG = spark.sql(
    "SELECT current_catalog()"
).first()[0]

SCHEMA = "default"

VOLUME_ROOT = "/Volumes/workspace/lab_sherhina_pn/raw"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_listings"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_listings"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_listings_features"

# --------------------------------------------------
# CITY PARAMETER
# --------------------------------------------------

city = dbutils.widgets.get("city").strip().lower()

config = {
    "url_slug": city.replace("_", "-"),
    "file_slug": city
}

print(f"Running pipeline for: {city}")

# --------------------------------------------------
# PATHS
# --------------------------------------------------

source_url = (
    "https://data.insideairbnb.com/"
    f"united-states/tx/{config['url_slug']}/"
    f"{SNAPSHOT}/data/listings.csv.gz"
)

raw_file = (
    f"{VOLUME_ROOT}/"
    f"{config['file_slug']}_{SNAPSHOT}_listings.csv.gz"
)

print(f"City: {city}")
print(f"Snapshot: {SNAPSHOT}")
print(f"Source: {source_url}")
print(f"Raw file: {raw_file}")
print(f"Bronze table: {BRONZE_TABLE}")
print(f"Silver table: {SILVER_TABLE}")
print(f"Gold table: {GOLD_TABLE}")

Running pipeline for: austin
City: austin
Snapshot: 2026-09-17
Source: https://data.insideairbnb.com/united-states/tx/austin/2026-09-17/data/listings.csv.gz
Raw file: /Volumes/workspace/lab_sherhina_pn/raw/austin_2026-09-17_listings.csv.gz
Bronze table: workspace.default.bronze_listings
Silver table: workspace.default.silver_listings


### Bronze layer

In [0]:
# --------------------------------------------------
# DOWNLOAD SOURCE FILE
# --------------------------------------------------

os.makedirs(VOLUME_ROOT, exist_ok=True)

if not os.path.exists(raw_file):
    print(f"Downloading {city}...")
    try:
        urllib.request.urlretrieve(
            source_url,
            raw_file
        )
    except Exception as e:
        print(f"Error downloading {city}: {e}") 

    print("Download completed.")
else:
    print("Source file already exists. Skipping download.")

# --------------------------------------------------
# READ RAW CSV
# --------------------------------------------------

bronze_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .option("inferSchema", False)
    .csv(raw_file)
)

# --------------------------------------------------
# ADD METADATA
# --------------------------------------------------

bronze_df = (
    bronze_df
    .withColumn("snapshot_date", F.lit(SNAPSHOT).cast("date"))
    .withColumn("city", F.lit(city))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

raw_count = bronze_df.count()

print(f"Loaded {raw_count:,} records.")
print(f"Columns: {len(bronze_df.columns)}")

display(bronze_df.limit(10))

Source file already exists. Skipping download.
Loaded 11,111 records.
Columns: 93


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,snapshot_date,city,ingestion_timestamp
741434727341232814,https://www.airbnb.com/rooms/741434727341232814,20260917204843,2026-09-19,previous scrape,Private bedroom in fantastic shared home,Keep it simple at this peaceful and centrally-located place.,null,https://a0.muscache.com/pictures/miso/Hosting-741434727341232814/original/f44da8ac-5629-456f-8d6e-f0f33cce9fc6.png,123504982,https://www.airbnb.com/users/show/123504982,1468248752934160688,https://www.airbnb.com/users/profile/1468248752934160688,Quinn,null,9,5,3,10,"Austin, TX",Friendly and active digital nomad.,null,null,null,f,null,https://a0.muscache.com/im/pictures/user/feeb1b1c-542b-4d1f-9f0e-210baec04362.jpg?aki_policy=profile_x_medium,null,1,null,None,t,t,null,78721,null,30.27271,-97.68135,Private room in home,Private room,1,null,2.5 baths,4,4,"[""Air conditioning"", ""Kitchen"", ""Washer"", ""Wifi"", ""TV"", ""Smoke alarm"", ""Carbon monoxide alarm"", ""Fire extinguisher"", ""Free parking on premises""]",$79.74,2026-09-19,2026-12-18,7176.93,79.74,"{""quote"": {""taxes"": null, ""currency"": ""USD"", ""date_match"": null, ""service_fee"": null, ""total_price"": ""7176.930000000000000000000000"", ""cleaning_fee"": null, ""is_available"": true, ""discount_amount"": null, ""price_per_night"": ""79.74366666666666666666666667"", ""nightly_subtotal"": null, ""is_monthly_priced"": true, ""discounted_subtotal"": ""2392.31"", ""raw_price_line_items"": [{""amount"": ""2470.50"", ""item_type"": ""other"", ""description"": ""Average monthly price"", ""price_string"": ""$2,470.50""}, {""amount"": ""-78.19"", ""item_type"": ""other"", ""description"": ""Airbnb monthly stay savings"", ""price_string"": ""-$78.19""}, {""amount"": ""2392.31"", ""item_type"": ""discounted_subtotal"", ""description"": ""Price after discount"", ""price_string"": ""$2,392.31""}], ""returned_checkin_date"": null, ""requested_checkin_date"": ""2026-09-19"", ""returned_checkout_date"": null, ""requested_checkout_date"": ""2026-12-18""}}",90,365,90,90,365,365,90.0,365.0,null,t,30,60,90,365,2026-09-19,0,0,0,104,0,0,0,null,null,null,null,null,null,null,null,null,null,null,1,0,1,0,null,2026-09-17,austin,2026-11-11T18:14:22.334Z
1775522910816348491,https://www.airbnb.com/rooms/1775522910816348491,20260917204843,2026-09-20,null,Apartment in Austin,Entire place apartment,null,https:/

In [0]:
# --------------------------------------------------
# WRITE BRONZE DELTA TABLE
# --------------------------------------------------

if not spark.catalog.tableExists(BRONZE_TABLE):

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("city", "snapshot_date")
        .saveAsTable(BRONZE_TABLE)
    )

else:

    # Replace only the partition for the current
    # city and snapshot, preserving other cities.

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .option("mergeSchema", "true")
        .saveAsTable(BRONZE_TABLE)
    )

print("Bronze ingestion completed.")

display(
    spark.table(BRONZE_TABLE)
    .filter(F.col("city") == city)
    .limit(10)
)

Bronze ingestion completed.


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,snapshot_date,city,ingestion_timestamp
741434727341232814,https://www.airbnb.com/rooms/741434727341232814,20260917204843,2026-09-19,previous scrape,Private bedroom in fantastic shared home,Keep it simple at this peaceful and centrally-located place.,null,https://a0.muscache.com/pictures/miso/Hosting-741434727341232814/original/f44da8ac-5629-456f-8d6e-f0f33cce9fc6.png,123504982,https://www.airbnb.com/users/show/123504982,1468248752934160688,https://www.airbnb.com/users/profile/1468248752934160688,Quinn,null,9,5,3,10,"Austin, TX",Friendly and active digital nomad.,null,null,null,f,null,https://a0.muscache.com/im/pictures/user/feeb1b1c-542b-4d1f-9f0e-210baec04362.jpg?aki_policy=profile_x_medium,null,1,null,None,t,t,null,78721,null,30.27271,-97.68135,Private room in home,Private room,1,null,2.5 baths,4,4,"[""Air conditioning"", ""Kitchen"", ""Washer"", ""Wifi"", ""TV"", ""Smoke alarm"", ""Carbon monoxide alarm"", ""Fire extinguisher"", ""Free parking on premises""]",$79.74,2026-09-19,2026-12-18,7176.93,79.74,"{""quote"": {""taxes"": null, ""currency"": ""USD"", ""date_match"": null, ""service_fee"": null, ""total_price"": ""7176.930000000000000000000000"", ""cleaning_fee"": null, ""is_available"": true, ""discount_amount"": null, ""price_per_night"": ""79.74366666666666666666666667"", ""nightly_subtotal"": null, ""is_monthly_priced"": true, ""discounted_subtotal"": ""2392.31"", ""raw_price_line_items"": [{""amount"": ""2470.50"", ""item_type"": ""other"", ""description"": ""Average monthly price"", ""price_string"": ""$2,470.50""}, {""amount"": ""-78.19"", ""item_type"": ""other"", ""description"": ""Airbnb monthly stay savings"", ""price_string"": ""-$78.19""}, {""amount"": ""2392.31"", ""item_type"": ""discounted_subtotal"", ""description"": ""Price after discount"", ""price_string"": ""$2,392.31""}], ""returned_checkin_date"": null, ""requested_checkin_date"": ""2026-09-19"", ""returned_checkout_date"": null, ""requested_checkout_date"": ""2026-12-18""}}",90,365,90,90,365,365,90.0,365.0,null,t,30,60,90,365,2026-09-19,0,0,0,104,0,0,0,null,null,null,null,null,null,null,null,null,null,null,1,0,1,0,null,2026-09-17,austin,2026-11-11T18:15:38.365Z
1775522910816348491,https://www.airbnb.com/rooms/1775522910816348491,20260917204843,2026-09-20,null,Apartment in Austin,Entire place apartment,null,https:/

## Silver layer

#### Column Selection

The original Airbnb dataset contains 92 columns, many of which are irrelevant to price analysis or contain redundant information. Following the exploratory data analysis, the pipeline retains a selected set of 26 columns covering listing identifiers, geographic information, host attributes, property characteristics, availability, reviews, and the target variable (price).

This selection preserves the variables identified during EDA as potentially useful for understanding and predicting listing prices, while excluding raw descriptions, URLs, and other unnecessary fields.

In [0]:
# --------------------------------------------------
# READ BRONZE DATA
# --------------------------------------------------

bronze = (
    spark.table(BRONZE_TABLE)
    .filter(
        (F.col("city") == city) &
        (F.col("snapshot_date") == SNAPSHOT)
    )
)

# --------------------------------------------------
# COLUMN SELECTION
# --------------------------------------------------

KEEP = [
    # Identifiers
    "id",
    "host_id",
    "city",

    # Geographic
    "neighbourhood_cleansed",
    "latitude",
    "longitude",

    # Host attributes
    "host_is_superhost",
    "calculated_host_listings_count",

    # Property characteristics
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms_text",
    "bedrooms",
    "beds",
    "amenities",

    # Availability and demand
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_365",
    "estimated_occupancy_l365d",

    # Reviews
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "reviews_per_month",

    # Target
    "price"
]

actual_keep = [
    column for column in KEEP
    if column in bronze.columns
]

required_columns = [
    "id",
    "city",
    "price",
    "room_type",
    "bedrooms",
    "neighbourhood_cleansed",
    "maximum_nights",
    "minimum_nights"
]

missing_required = [
    column for column in required_columns
    if column not in bronze.columns
]

if missing_required:
    raise ValueError(
        f"Required source columns missing: {missing_required}"
    )

listings = bronze.select(
    *actual_keep,
    "snapshot_date"
)

print(
    f"Selected {len(actual_keep)} EDA columns."
)

display(listings.limit(10))

Selected 26 EDA columns.


id,host_id,city,neighbourhood_cleansed,latitude,longitude,host_is_superhost,calculated_host_listings_count,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,amenities,minimum_nights,maximum_nights,availability_30,availability_365,estimated_occupancy_l365d,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,reviews_per_month,price,snapshot_date
741434727341232814,123504982,austin,78721,30.27271,-97.68135,f,1,Private room in home,Private room,1,2.5 baths,4,4,"[""Air conditioning"", ""Kitchen"", ""Washer"", ""Wifi"", ""TV"", ""Smoke alarm"", ""Carbon monoxide alarm"", ""Fire extinguisher"", ""Free parking on premises""]",90,365,30,365,0,0,null,null,null,null,$79.74,2026-09-17
1775522910816348491,714096542,austin,78705,30.2935592,-97.7404389,f,1,Entire rental unit,Entire home/apt,3,1 bath,1,1,"[""Air conditioning"", ""Kitchen"", ""Wifi"", ""TV"", ""Smoke alarm"", ""Electric stove"", ""Coffee maker: Keurig coffee machine"", ""Free parking on premises""]",1,365,17,337,6,1,5.0,5.0,5.0,1,$158.00,2026-09-17
795719559342015447,27228465,austin,78737,30.2184056,-98.0214838,f,1,Entire guesthouse,Entire home/apt,4,1 bath,1,2,"[""Air conditioning"", ""Kitchen"", ""Outdoor dining area"", ""Wifi"", ""TV"", ""Smoke alarm"", ""Dedicated workspace"", ""First aid kit"", ""Fire extinguisher"", ""Free parking on premises""]",2,2,28,88,0,0,null,null,null,null,null,2026-09-17
755672804161441366,78455367,austin,78705,30.28369,-97.74629,f,11,Room in boutique hotel,Private room,2,1 private bath,1,1,"[""Air conditioning"", ""Heating"", ""Wifi"", ""Pack \u2019n play/Travel crib - available upon request"", ""TV"", ""Paid parking on premises"", ""Pets allowed"", ""Carbon monoxide alarm"", ""Hair dryer"", ""Fire extinguisher"", ""Pool""]",1,999,16,296,0,0,null,null,null,null,$428.80,2026-09-17
613088323255036264,386381014,austin,78701,30.26812,-97.73742,f,3,Entire rental unit,Entire home/apt,4,1 bath,1,2,"[""Air conditioning"", ""Iron"", ""Heating"", ""Elevator"", ""Bed linens"", ""Wifi"", ""TV"", ""Smoke alarm"", ""Gym"", ""Paid parking off premises"", ""Essentials"", ""Body soap"", ""Hair dryer"", ""Shampoo"", ""Host greets you""]",1,365,30,363,0,163,4.75,4.88,4.75,3.06,$499.00,2026-09-17
1100736390821200576,56268853,austin,78703,30.27477,-97.75445,f,1,Camper/RV,Entire home/apt,1,Half-bath,1,1,[],1,365,29,269,0,0,null,null,null,null,$500.00,2026-09-17
1630500606678657796,748606753,austin,78701,30.27051,-97.74727,f,1,Entire rental unit,Entire home/apt,2,1 bath,1,1,"[""Air conditioning"", ""Kitchen"", ""Wifi"", ""TV"", ""BBQ grill"", ""Paid parking on premises"", ""Dedicated workspace"", ""Pets allowed"", ""Free parking on premises"", ""Pool""]",1,365,30,90,0,0,null,null,null,null,$649.00,2026-09-17
1521869937587597751,406414334,austin,78701,30.27027,-97.74699,f,4,Room in hotel,Private room,4,1 private bath,1,2,"[""Air conditioning"", ""Outdoor dining area"", ""Heating"", ""Wifi"", ""Smoke alarm"", ""Dedicated workspace"", ""Iron"", ""Hot tub"", ""TV"", ""First aid kit"", ""Hair dryer"", ""Pool"", ""BBQ grill"", ""Paid parking on premises"", ""Self check-in"", ""Building staff"", ""Carbon monoxide alarm"", ""Exercise equipment"", ""Cooking basics"", ""Kitchen"", ""Fire pit"", ""Fire extinguisher"", ""Free parking on premises""]",2,28,30,362,0,0,null,null,null,null,$339.50,2026-09-17
1754485350963136016,318559,austin,78751,30.3106693,-97.7304413,f,1,Private room in rental unit,Private room,2,2.5 shared baths,1,1,"[""Air conditioning"", ""Washer"", ""Wifi"", ""TV""]",30,365,30,364,0,0,null,null,null,null,$63.09,2026-09-17
1485032739241135557,391993902,austin,78751,30.30663,-97.717,t,1,Entire rental unit,Entire home/apt,2,1 bath,0,1,"[""Air conditioning"", ""Kitchen"", ""Wifi"", ""TV"", ""Smoke alarm"", ""Dedicated workspace"", ""Fire extinguisher"", ""Free parking on premises""]",30,365,0,334,0,0,null,null,null,null,null,2026-09-17


#### Data Type Conversion and Feature Extraction

The EDA identified several columns whose original string representation prevents meaningful statistical analysis. The pipeline converts the price column into a numeric value, transforms the superhost indicator into a boolean, and parses the amenities column into an array of strings.

The `bathrooms_text` column is also transformed into two separate features: a numeric bathroom count and a boolean indicator showing whether the bathroom is shared. This preserves both the quantitative and categorical information contained in the original column.

In [0]:
# --------------------------------------------------
# PRICE
# --------------------------------------------------

typed = listings.withColumn(
    "price",
    F.regexp_replace(
        F.col("price"),
        r"[$,]",
        ""
    ).cast("double")
)

# --------------------------------------------------
# BOOLEAN COLUMNS
# --------------------------------------------------

typed = typed.withColumn(
    "host_is_superhost",
    F.col("host_is_superhost") == "t"
)

# --------------------------------------------------
# AMENITIES
# --------------------------------------------------

typed = typed.withColumn(
    "amenities",
    F.from_json(
        F.col("amenities"),
        T.ArrayType(T.StringType())
    )
)

# --------------------------------------------------
# BATHROOMS
# --------------------------------------------------

bathroom_number = F.regexp_extract(
    F.col("bathrooms_text"),
    r"^([0-9]+\.?[0-9]*)",
    1
)

typed = (
    typed
    .withColumn(
        "bathrooms",
        F.when(
            F.col("bathrooms_text").isNull(),
            None
        )
        .when(
            bathroom_number != "",
            bathroom_number.cast("double")
        )
        .otherwise(F.lit(0.5))
    )
    .withColumn(
        "bathroom_shared",
        F.col("bathrooms_text").rlike("(?i)shared")
    )
    .drop("bathrooms_text")
)

print("Initial type conversions completed.")

Initial type conversions completed.


#### Numeric Type Conversion

The EDA identified several numerical attributes stored as strings in the original dataset. These columns must be converted into appropriate numeric types before calculating distributions, correlations, and other statistical measures.

Coordinates and continuous measurements are converted to double precision, while counts and integer-valued attributes are converted to integers. This includes property capacity, bedroom and bed counts, minimum and maximum nights, availability, occupancy, and review-related metrics.

In [0]:
# --------------------------------------------------
# NUMERIC TYPES
# --------------------------------------------------

NUMERIC_TYPES = {
    "id": "long",
    "host_id": "long",
    "latitude": "double",
    "longitude": "double",

    "calculated_host_listings_count": "integer",

    "accommodates": "integer",
    "bedrooms": "integer",
    "beds": "integer",

    "minimum_nights": "integer",
    "maximum_nights": "integer",

    "availability_30": "integer",
    "availability_365": "integer",
    "estimated_occupancy_l365d": "double",

    "number_of_reviews": "integer",
    "review_scores_rating": "double",
    "review_scores_cleanliness": "double",
    "review_scores_location": "double",
    "reviews_per_month": "double"
}

for column_name, data_type in NUMERIC_TYPES.items():

    if column_name in typed.columns:

        typed = typed.withColumn(
            column_name,
            F.col(column_name).cast(data_type)
        )

print("Numeric type conversions completed.")

Numeric type conversions completed.


#### Missing-Value Handling

The exploratory analysis showed that missing values in the Airbnb dataset are often structural rather than random. In particular, missing bedroom counts are associated with studio properties, missing review scores indicate listings without previous reviews, and missing neighborhood values are concentrated in Fort Worth.

To preserve this information, the pipeline creates an `is_studio` indicator before replacing missing bedroom counts with zero. It also creates a `has_reviews` indicator while retaining null review scores and review frequency values.

Missing neighborhood values in Fort Worth are replaced with the placeholder "Unknown - Fort Worth". This prevents the loss of listings when geographic information is unavailable. The target variable, price, is not imputed; listings without a valid price are removed during cleaning.

In [0]:
# --------------------------------------------------
# BEDROOMS
# --------------------------------------------------

# Preserve information about originally missing values
typed = typed.withColumn(
    "bedrooms_was_missing",
    F.col("bedrooms").isNull()
)

# Missing bedrooms in entire homes/apartments
# are treated as studio listings
typed = typed.withColumn(
    "is_studio",
    (
        F.col("bedrooms_was_missing") &
        (F.col("room_type") == "Entire home/apt")
    )
)

# Replace missing bedroom counts with zero
typed = typed.withColumn(
    "bedrooms",
    F.coalesce(
        F.col("bedrooms"),
        F.lit(0)
    ).cast("integer")
)

# --------------------------------------------------
# REVIEWS
# --------------------------------------------------

# Preserve missing review scores and review frequency.
# Add a flag identifying listings with previous reviews.

typed = typed.withColumn(
    "has_reviews",
    F.when(
        F.col("number_of_reviews").isNull(),
        None
    ).otherwise(
        F.col("number_of_reviews") > 0
    )
)

# --------------------------------------------------
# NEIGHBORHOODS
# --------------------------------------------------

# Fort Worth has substantial neighborhood missingness.
# Preserve these listings using an explicit placeholder.

typed = typed.withColumn(
    "neighbourhood_cleansed",
    F.when(
        (
            (F.col("city") == "fort_worth") &
            F.col("neighbourhood_cleansed").isNull()
        ),
        F.lit("Unknown - Fort Worth")
    ).otherwise(
        F.col("neighbourhood_cleansed")
    )
)

print("Missing-value handling completed.")

display(
    typed.select(
        "city",
        "room_type",
        "bedrooms",
        "bedrooms_was_missing",
        "is_studio",
        "number_of_reviews",
        "review_scores_rating",
        "has_reviews",
        "neighbourhood_cleansed"
    ).limit(10)
)

Missing-value handling completed.


city,room_type,bedrooms,bedrooms_was_missing,is_studio,number_of_reviews,review_scores_rating,has_reviews,neighbourhood_cleansed
austin,Private room,4,false,false,0,null,false,78721
austin,Entire home/apt,1,false,false,1,5.0,true,78705
austin,Entire home/apt,1,false,false,0,null,false,78737
austin,Private room,1,false,false,0,null,false,78705
austin,Entire home/apt,1,false,false,163,4.75,true,78701
austin,Entire home/apt,1,false,false,0,null,false,78703
austin,Entire home/apt,1,false,false,0,null,false,78701
austin,Private room,1,false,false,0,null,false,78701
austin,Private room,1,false,false,0,null,false,78751
austin,Entire home/apt,0,false,false,0,null,false,78751


#### Outlier Classification

The EDA identified three categories of listings that do not represent the intended short-term rental market:

- Impossible prices: prices less than or equal to zero or greater than $1,500.
- Impossible maximum stays: listings with `maximum_nights` greater than 1,125 days, exceeding the platform limit identified in the analysis.
- Long-term rentals: listings with `minimum_nights` greater than 90 days.

The pipeline assigns an issue category to each listing before removing invalid records. This makes it possible to inspect the number of affected listings and understand the impact of the cleaning rules.

In [0]:
# --------------------------------------------------
# ISSUE CLASSIFICATION
# --------------------------------------------------

typed = typed.withColumn(
    "issue",

    F.when(
        (F.col("price") <= 0) |
        (F.col("price") > 1500),
        "impossible_price"
    )

    .when(
        F.col("maximum_nights") > 1125,
        "impossible_max_nights"
    )

    .when(
        F.col("minimum_nights") > 90,
        "long_term"
    )

    .otherwise("ok")
)

# --------------------------------------------------
# ISSUE DISTRIBUTION
# --------------------------------------------------

display(
    typed.groupBy("issue")
    .count()
    .orderBy("issue")
)

issue,count
impossible_max_nights,4
impossible_price,207
long_term,50
ok,10850


#### Data Cleaning and Row Filtering

After classifying invalid listings, the pipeline removes records that do not meet the requirements of the short-term rental price analysis.

Listings are excluded if their price is missing, less than or equal to zero, or greater than $1,500. Listings with `maximum_nights` greater than 1,125 or `minimum_nights` greater than 90 are also removed.

Missing values in `maximum_nights` and `minimum_nights` are retained because the EDA does not identify them as grounds for exclusion. Duplicate records are removed using the combination of city, snapshot date, and listing ID as the unique listing key.

These operations produce the cleaned Silver dataset used in subsequent analysis and model development.

In [0]:
# --------------------------------------------------
# REMOVE INVALID LISTINGS
# --------------------------------------------------

# Remove:
# - Missing prices
# - Prices <= 0
# - Prices > $1500
# - maximum_nights > 1125
# - minimum_nights > 90
#
# Retain null minimum_nights and maximum_nights.

cleaned = typed.filter(
    F.col("price").isNotNull() &
    (F.col("price") > 0) &
    (F.col("price") <= 1500) &
    (
        F.col("maximum_nights").isNull() |
        (F.col("maximum_nights") <= 1125)
    ) &
    (
        F.col("minimum_nights").isNull() |
        (F.col("minimum_nights") <= 90)
    )
)

# --------------------------------------------------
# REMOVE DUPLICATES
# --------------------------------------------------

cleaned = cleaned.dropDuplicates(
    ["city", "snapshot_date", "id"]
)

# --------------------------------------------------
# CLEANING SUMMARY
# --------------------------------------------------

before_count = typed.count()
after_count = cleaned.count()

print(f"Before cleaning: {before_count:,}")
print(f"After cleaning:  {after_count:,}")
print(f"Removed:         {before_count - after_count:,}")

Before cleaning: 11,111
After cleaning:  9,759
Removed:         1,352


#### Silver Layer Storage

The cleaned and transformed listings are stored in the Silver layer as a Delta table. This layer contains the selected analytical features, corrected data types, extracted bathroom attributes, missing-value indicators, and filtered listing records.

The table is partitioned by city and snapshot date. This allows the same parameterized pipeline to process multiple cities and snapshots while updating the relevant partition without replacing data belonging to other cities.

The Silver table provides the cleaned dataset for subsequent exploratory analysis, feature engineering, and price prediction.


In [0]:
# --------------------------------------------------
# WRITE SILVER DELTA TABLE
# --------------------------------------------------

if not spark.catalog.tableExists(SILVER_TABLE):

    (
        cleaned.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("city", "snapshot_date")
        .saveAsTable(SILVER_TABLE)
    )

else:

    (
        cleaned.write
        .format("delta")
        .mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .option("mergeSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )

print("Silver transformation completed.")

Silver transformation completed.


#### Silver Layer Validation

After writing the cleaned data, the pipeline reads the Silver table for the currently selected city and snapshot date. A set of data quality checks is then performed to verify that the transformation and cleaning operations were successful.

The checks verify that listing IDs are present and unique, prices are valid and non-null, and no listings violate the maximum- or minimum-night restrictions defined during EDA.

The pipeline also reports the number of records in Bronze and Silver, allowing the effect of the cleaning process to be evaluated. Assertions ensure that the pipeline fails if any of the required data quality conditions are violated.

In [0]:
# --------------------------------------------------
# READ SILVER
# --------------------------------------------------

silver = (
    spark.table(SILVER_TABLE)
    .filter(
        (F.col("city") == city) &
        (F.col("snapshot_date") == SNAPSHOT)
    )
)

# --------------------------------------------------
# BASIC COUNTS
# --------------------------------------------------

bronze_count = bronze.count()
silver_count = silver.count()

print(f"City: {city}")
print(f"Bronze records: {bronze_count:,}")
print(f"Silver records: {silver_count:,}")
print(f"Removed records: {bronze_count - silver_count:,}")

display(
    silver.groupBy("city", "snapshot_date")
    .count()
)

# --------------------------------------------------
# NULL ID CHECK
# --------------------------------------------------

null_ids = silver.filter(
    F.col("id").isNull()
).count()

print(f"Null IDs: {null_ids}")

# --------------------------------------------------
# DUPLICATE CHECK
# --------------------------------------------------

duplicates = (
    silver
    .groupBy("city", "snapshot_date", "id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate listings: {duplicates}")

# --------------------------------------------------
# MISSING PRICE CHECK
# --------------------------------------------------

missing_prices = silver.filter(
    F.col("price").isNull()
).count()

print(f"Missing prices: {missing_prices}")

# --------------------------------------------------
# INVALID PRICE CHECK
# --------------------------------------------------

invalid_prices = silver.filter(
    (F.col("price") <= 0) |
    (F.col("price") > 1500)
).count()

print(f"Invalid prices: {invalid_prices}")

# --------------------------------------------------
# MAXIMUM NIGHTS CHECK
# --------------------------------------------------

invalid_max_nights = silver.filter(
    F.col("maximum_nights") > 1125
).count()

print(f"Invalid maximum nights: {invalid_max_nights}")

# --------------------------------------------------
# MINIMUM NIGHTS CHECK
# --------------------------------------------------

long_term = silver.filter(
    F.col("minimum_nights") > 90
).count()

print(f"Long-term listings: {long_term}")

# --------------------------------------------------
# ASSERTIONS
# --------------------------------------------------

assert null_ids == 0, \
    "Null listing IDs found!"

assert duplicates == 0, \
    "Duplicate listings found!"

assert missing_prices == 0, \
    "Missing prices found!"

assert invalid_prices == 0, \
    "Invalid prices found!"

assert invalid_max_nights == 0, \
    "Listings exceeding maximum_nights limit found!"

assert long_term == 0, \
    "Long-term listings found!"

print("All data quality checks passed.")

# --------------------------------------------------
# FINAL RESULT
# --------------------------------------------------

display(silver.limit(20))

City: austin
Bronze records: 11,111
Silver records: 9,759
Removed records: 1,352


city,snapshot_date,count
austin,2026-09-17,9759


Null IDs: 0
Duplicate listings: 0
Missing prices: 0
Invalid prices: 0
Invalid maximum nights: 0
Long-term listings: 0
All data quality checks passed.


id,host_id,city,neighbourhood_cleansed,latitude,longitude,host_is_superhost,calculated_host_listings_count,property_type,room_type,accommodates,bedrooms,beds,amenities,minimum_nights,maximum_nights,availability_30,availability_365,estimated_occupancy_l365d,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,reviews_per_month,price,snapshot_date,bathrooms,bathroom_shared,bedrooms_was_missing,is_studio,has_reviews,issue
755672804161441366,78455367,austin,78705,30.28369,-97.74629,false,11,Room in boutique hotel,Private room,2,1,1,"List(Air conditioning, Heating, Wifi, Pack ’n play/Travel crib - available upon request, TV, Paid parking on premises, Pets allowed, Carbon monoxide alarm, Hair dryer, Fire extinguisher, Pool)",1,999,16,296,0.0,0,null,null,null,null,428.8,2026-09-17,1.0,false,false,false,false,ok
1434253263413807084,41945984,austin,78745,30.2176426,-97.77687929999999,true,1,Entire home,Entire home/apt,4,2,2,"List(Air conditioning, Blender, Extra pillows and blankets, Central heating, Stainless steel single oven, Refrigerator, Outdoor furniture, Books and reading material, Dishes and silverware, Wifi, Dishwasher, Smoke alarm, Dedicated workspace, Conditioner, Exterior security cameras on property, Baking sheet, Dining table, Iron, Private backyard – Fully fenced, Outdoor shower, Shampoo, Coffee, Ceiling fan, Shower gel, Hangers, Hot water, TV, Smart lock, Clothing storage, Freezer, First aid kit, Hair dryer, Room-darkening shades, Pool, Cleaning products, Free street parking, Wine glasses, Coffee maker: drip coffee maker, Self check-in, Carbon monoxide alarm, Cooking basics, Microwave, Kitchen, Washer, Bed linens, Free dryer – In building, Bathtub, Gas stove, Body soap, Fire extinguisher, Free parking on premises, Hot water kettle)",2,1125,7,231,255.0,60,4.95,4.93,4.77,4.21,393.5,2026-09-17,2.5,false,false,false,true,ok
645540099314291356,254423869,austin,78748,30.16025,-97.83425,false,1,Entire home,Entire home/apt,6,3,3,"List(Blender, Extra pillows and blankets, Drying rack for clothing, Toaster, Outdoor dining area, Heating, Refrigerator, Outdoor furniture, Dishes and silverware, Coffee maker, Wifi, Dishwasher, Smoke alarm, Keypad, Dedicated workspace, Conditioner, Dining table, Baking sheet, Iron, Backyard, Shampoo, Central air conditioning, Ceiling fan, Coffee, 65 inch HDTV with Amazon Prime Video, Netflix, premium cable, Shower gel, Hangers, Hot water, Free washer – In unit, Clothing storage, Freezer, Children’s books and toys, Hair dryer, First aid kit, Private patio or balcony, Pack ’n play/Travel crib, Wine glasses, Oven, Essentials, Self check-in, Carbon monoxide alarm, Long term stays allowed, Cooking basics, Microwave, Kitchen, Free dryer – In unit, Bed linens, Stove, Children’s dinnerware, Body soap, Fire extinguisher, Free parking on premises, Hot water kettle)",3,1125,17,167,42.0,37,4.65,4.68,4.78,0.78,152.8,2026-09-17,2.5,false,false,false,true,ok
578115945551215417,114174610,austin,78758,30.38327,-97.70035,false,3,Entire home,Entire home/apt,6,3,5,"List(Clothing storage: walk-in closet, Outdoor dining area, Central heating, Toaster, Refrigerator, Outdoor furniture, Dishes and silverware, Wifi, Dishwasher, Smoke alarm, Keypad, Dedicated workspace, Conditioner, Dining table, Baking sheet, Stainless steel oven, Private backyard – Fully fenced, Shampoo, Free parking garage on premises, Central air conditioning, Shower gel, Hangers, Hot water, Free washer – In unit, Pets allowed, Freezer, Hair dryer, Private patio or balcony, 55 inch HDTV, Room-darkening shades, Indoor fireplace, Cleaning products, Wine glasses, Coffee maker: drip coffee maker, Self check-in, Essentials, Carbon monoxide alarm, Single level home, Long term stays allowed, Cooking basics, Microwave, Private entrance, Kitchen, Free dryer – In unit, Bed linens, Stove, Bathtub, Fire extinguisher, Hot water kettle)",2,365,0,177,42.0,32,4.94,4.84,4.81,0.9,221.03,2026-09-17,2.0,false,false,false,true,ok
1438768242476

### Gold Layer


#### Selection of Analytics and Model Attributes

This step selects the columns required for the Gold dataset from the cleaned Silver records.

The selected attributes cover listing characteristics, location, host information, availability, reviews, and price. The listing identifier is retained to support record traceability and comparison between actual and predicted prices.

The original `bathrooms_text` field is not included because it was transformed into numeric `bathrooms` and boolean `bathroom_shared` attributes during Silver-layer processing.

The dataset also retains `is_studio` and `has_reviews`, which represent information derived from missing-value analysis in the EDA.

Columns used exclusively for intermediate cleaning operations are excluded from the Gold dataset.


In [0]:

GOLD_COLUMNS = [
    # Identifiers and metadata
    "id",
    "host_id",
    "city",
    "snapshot_date",

    # Location
    "neighbourhood_cleansed",
    "latitude",
    "longitude",

    # Host characteristics
    "host_is_superhost",
    "calculated_host_listings_count",

    # Property characteristics
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bathroom_shared",
    "bedrooms",
    "beds",
    "is_studio",
    "amenities",

    # Booking and availability
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_365",
    "estimated_occupancy_l365d",

    # Reviews
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "reviews_per_month",
    "has_reviews",

    # Prediction target
    "price"
]

missing_columns = [
    column for column in GOLD_COLUMNS
    if column not in silver.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in Silver table: {missing_columns}"
    )

gold_df = silver.select(*GOLD_COLUMNS)

print(f"Gold columns: {len(gold_df.columns)}")
print(f"Gold records: {gold_df.count():,}")

display(gold_df.limit(10))


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5667462904610420>, line 50
      2 GOLD_COLUMNS = [
      3     # Identifiers and metadata
      4     "id",
   (...)     45     "price"
     46 ]
     48 missing_columns = [
     49     column for column in GOLD_COLUMNS
---> 50     if column not in silver.columns
     51 ]
     53 if missing_columns:
     54     raise ValueError(
     55         f"Missing columns in Silver table: {missing_columns}"
     56     )

NameError: name 'silver' is not defined


#### Persisting the Feature Dataset

After selecting the required attributes, the pipeline writes the resulting dataset to the `gold_listings_features` Delta table.

The table is partitioned by `city` and `snapshot_date`. This organization supports processing multiple cities and snapshots while allowing individual city partitions to be updated without replacing the entire dataset.

For the initial execution, the table is created with the selected schema. For subsequent executions, dynamic partition overwrite is used so that processing one city does not remove records belonging to another city.

This makes the Gold-layer transformation reusable for both training cities and the unseen test city.


In [0]:

if spark.catalog.tableExists(GOLD_TABLE):

    # Replace only the partitions for the city and snapshot
    # currently being processed.
    (
        gold_df.write
        .format("delta")
        .mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .saveAsTable(GOLD_TABLE)
    )

else:

    # Create the Gold table on the first execution.
    (
        gold_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("city", "snapshot_date")
        .saveAsTable(GOLD_TABLE)
    )

print(f"Gold table successfully updated: {GOLD_TABLE}")



#### Data Quality Validation

After writing the Gold dataset, the pipeline performs validation checks to verify that the resulting table contains valid and consistent records.

The checks include:

- Confirming that the Gold table contains records for the current city.
- Checking for missing listing identifiers.
- Checking for duplicate listings within the same city and snapshot.
- Checking for missing or invalid target prices.
- Confirming that the expected Gold columns are present.

These checks help ensure that the Gold dataset is suitable for downstream analytics and machine learning.


In [0]:

gold_saved = (
    spark.table(GOLD_TABLE)
    .filter(F.col("city") == city)
)

gold_count = gold_saved.count()

null_ids = gold_saved.filter(
    F.col("id").isNull()
).count()

duplicate_ids = (
    gold_saved
    .groupBy("city", "snapshot_date", "id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_prices = gold_saved.filter(
    F.col("price").isNull()
).count()

invalid_prices = gold_saved.filter(
    (F.col("price") <= 0) | (F.col("price") > 1500)
).count()

print(f"Gold table: {GOLD_TABLE}")
print(f"City: {city}")
print(f"Total records: {gold_count:,}")
print(f"Null listing IDs: {null_ids}")
print(f"Duplicate listing groups: {duplicate_ids}")
print(f"Null prices: {null_prices}")
print(f"Invalid prices: {invalid_prices}")

assert gold_count > 0, "Gold table contains no records for this city."
assert null_ids == 0, "Gold table contains null listing IDs."
assert duplicate_ids == 0, "Gold table contains duplicate listings."
assert null_prices == 0, "Gold table contains null prices."
assert invalid_prices == 0, "Gold table contains invalid prices."

print("Gold data quality checks passed.")

display(gold_saved.limit(10))
